# load data

In [9]:
import pandas as pd
from pandas_datareader import data

reddit_data = pd.read_csv("reddit_combined.csv")
reddit_data["date"] = pd.to_datetime(reddit_data['date']).dt.tz_localize(None)
reddit_data["date"] = pd.to_datetime(reddit_data['date']).dt.date

# get twitter data
twitter_data = pd.read_csv("twitter_combined.csv")
twitter_data["date"] = pd.to_datetime(twitter_data['Datetime']).dt.tz_localize(None)
twitter_data["date"] = pd.to_datetime(twitter_data['Datetime']).dt.date

# get stocktwits data
stocktwits_data = pd.read_csv("stocktwits_combined.csv")
stocktwits_data["date"] = pd.to_datetime(stocktwits_data['date_created']).dt.tz_localize(None)
stocktwits_data["date"] = pd.to_datetime(stocktwits_data['date_created']).dt.date

# # get news data
# news_data = pd.read_csv("./news/cleaned_AAPL_sentiment.csv")
# news_data["date"] = pd.to_datetime(reddit_data['date'])
# get stocktwits data
news_data = pd.read_csv("news_combined.csv")
news_data["date"] = pd.to_datetime(news_data['date']).dt.tz_localize(None)
news_data["date"] = pd.to_datetime(news_data['date']).dt.date


# standardize scores 

In [10]:
def transform_sentiment(df):
    for index, row in df.iterrows():
        if row['flair_sentiment'] == "NEGATIVE":
            df.at[index,"flair_sentiment_score"] = 0 - row['flair_sentiment_score']
        if row['finbert_sentiment'] == "negative":
            df.at[index,"finbert_score"] = 0 - row['finbert_score']
        if row['finbert_sentiment'] == "neutral":
            df.at[index,"finbert_score"] = 0
    return df

reddit_data = transform_sentiment(reddit_data)
twitter_data = transform_sentiment(twitter_data)
stocktwits_data = transform_sentiment(stocktwits_data)
news_data = transform_sentiment(news_data)



# correlation test 2 just weekdays drop weekends all stocks

In [61]:
uniqueticker = ['AAPL', 'AMZN', 'ARKK', 'GOOGL', 'MSFT', 'RUI', 'TSLA', 'VXRT']

def dropall(tickername,reddit_data,twitter_data,stocktwits_data,news_data):
    # get panel data
    start_date = '2020-04-29'
    end_date = '2021-10-10'
    panel_data = data.DataReader(tickername,'yahoo', start_date, end_date)
    panel_data["date"] = panel_data.index
    panel_data["date"] = pd.to_datetime(panel_data['date'])

    # groupby dates then average
    reddit_groupby = reddit_data[reddit_data['ticker'] == tickername].groupby(["date"]).mean()
    twitter_groupby = twitter_data[twitter_data['ticker'] == tickername].groupby(["date"]).mean()
    stocktwits_groupby = stocktwits_data[stocktwits_data['ticker'] == tickername].groupby(["date"]).mean()
    news_groupby = news_data[news_data['ticker'] == tickername].groupby(["date"]).mean()

    reddit_merge2=pd.merge(reddit_groupby,panel_data, how='inner', left_index=True, right_index=True)
    reddit_merge2.corr()[["flair_sentiment_score","vader_score","finbert_score"]]

    twitter_merge2=pd.merge(twitter_groupby,panel_data, how='inner', left_index=True, right_index=True)
    twitter_merge2.corr()[["flair_sentiment_score","vader_score","finbert_score"]]

    stocktwits_groupby["date"] = stocktwits_groupby.index
    stocktwits_groupby["date"] = pd.to_datetime(stocktwits_groupby['date'])
    stocktwits_merge2 = pd.merge(stocktwits_groupby,panel_data,how='inner', left_index=True, right_index=True)
    stocktwits_merge2.corr()[["flair_sentiment_score","vader_score","finbert_score"]]
    
    # merging each platform with the data
    news_merge2 = pd.merge(news_groupby,panel_data, how='inner', left_index=True, right_index=True)
    news_merge2.corr()[["flair_sentiment_score","vader_score","finbert_score"]]
    
    # create dataframe
    columns = ["flair_sentiment_score","vader_score","finbert_score","platform","ticket"]
    dropall = []
    dropall.append(reddit_merge2.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[6].tolist())
    dropall.append(twitter_merge2.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[9].tolist())
    dropall.append(stocktwits_merge2.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[8].tolist())
    dropall.append(news_merge2.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[4].tolist())
    dropall[0].append("reddit")
    dropall[0].append(tickername)
    dropall[1].append("twitter")
    dropall[1].append(tickername)
    dropall[2].append("stocktwits")
    dropall[2].append(tickername)
    dropall[3].append("news")
    dropall[3].append(tickername)
    dropall = pd.DataFrame(dropall,columns=columns)
#     dropall = dropall.set_index("platform")
    return dropall



In [62]:
i = 0
for ticket in uniqueticker:
    try:
        if i == 0:
            dropall_df = dropall(ticket,reddit_data,twitter_data,stocktwits_data,news_data)
            i += 1
        else:
            dropall_df_next = dropall(ticket,reddit_data,twitter_data,stocktwits_data,news_data)
            dropall_df = pd.concat([dropall_df, dropall_df_next])
    except:
        pass

In [63]:
dropall_df.to_csv("correlation_dropall.csv",index=False)

# correlation test 3 just weekdays drop weekends all stocks

In [129]:
from datetime import date
import calendar

def weighted_weekends(data):
    weighted_flair = weighted_vader = weighted_finbert = 0
    new_df = []
    for index, row in data.iterrows():
        day_name = row["day_name"]
        if (day_name == "Saturday"): # saturday
            weighted_flair += row['flair_sentiment_score'] * 0.2
            weighted_vader += row['vader_score'] * 0.2
            weighted_finbert += row['finbert_score'] * 0.2
            new_df.append([index, row['flair_sentiment_score'], row['vader_score'], row['finbert_score']])
        elif (day_name == "Sunday"): # sunday
            weighted_flair += row['flair_sentiment_score'] * 0.3
            weighted_vader += row['vader_score'] * 0.3
            weighted_finbert += row['finbert_score'] * 0.3
            new_df.append([index, row['flair_sentiment_score'], row['vader_score'], row['finbert_score']])
        elif (day_name == "Monday"): # monday
            weighted_flair += row['flair_sentiment_score'] * 0.5
            weighted_vader += row['vader_score'] * 0.5
            weighted_finbert += row['finbert_score'] * 0.5
            new_df.append([index, weighted_flair, weighted_vader, weighted_finbert])
            weighted_flair = weighted_vader = weighted_finbert = 0
            
        else:
            new_df.append([index, row['flair_sentiment_score'], row['vader_score'], row['finbert_score']])
    return new_df



def weighted_drop(tickername,reddit_data,twitter_data,stocktwits_data,news_data):
    # get panel data
    start_date = '2020-04-29'
    end_date = '2021-10-10'
    panel_data = data.DataReader(tickername,'yahoo', start_date, end_date)
    panel_data["date"] = panel_data.index
    panel_data["date"] = pd.to_datetime(panel_data['date'])

    # groupby dates then average
    reddit_groupby = reddit_data[reddit_data['ticker'] == tickername].groupby(["date"]).mean()
    twitter_groupby = twitter_data[twitter_data['ticker'] == tickername].groupby(["date"]).mean()
    stocktwits_groupby = stocktwits_data[stocktwits_data['ticker'] == tickername].groupby(["date"]).mean()
    news_groupby = news_data[news_data['ticker'] == tickername].groupby(["date"]).mean()
    
    # adding days inside 
    reddit_groupby["date"] = reddit_groupby.index
    reddit_groupby["date"] = pd.to_datetime(reddit_groupby['date'])
    for index, row in reddit_groupby.iterrows():
        date = pd.to_datetime(row["date"])
        reddit_groupby.at[index,"day_name"]  = calendar.day_name[date.weekday()]

    twitter_groupby["date"] = twitter_groupby.index
    twitter_groupby["date"] = pd.to_datetime(twitter_groupby['date'])
    for index, row in twitter_groupby.iterrows():
        date = pd.to_datetime(row["date"])
        twitter_groupby.at[index,"day_name"]  = calendar.day_name[date.weekday()]

    stocktwits_groupby["date"] = stocktwits_groupby.index
    stocktwits_groupby["date"] = pd.to_datetime(stocktwits_groupby['date'])
    for index, row in stocktwits_groupby.iterrows():
        date = pd.to_datetime(row["date"])
        stocktwits_groupby.at[index,"day_name"]  = calendar.day_name[date.weekday()]

    news_groupby["date"] = news_groupby.index
    news_groupby["date"] = pd.to_datetime(news_groupby['date'])
    for index, row in news_groupby.iterrows():
        date = pd.to_datetime(row["date"])
        news_groupby.at[index,"day_name"]  = calendar.day_name[date.weekday()]
    

    # using the weighted sentiment function    

    reddit_weight = weighted_weekends(reddit_groupby)
    reddit_weighted = pd.DataFrame(reddit_weight, columns=["date","flair_sentiment_score","vader_score","finbert_score"])
    reddit_weighted['date'] = reddit_weighted['date'].astype(str)
    panel_data['date'] = panel_data['date'].astype(str)
    reddit_merge3 = pd.merge(reddit_weighted,panel_data, on="date")
    reddit_merge3.corr()[["flair_sentiment_score","vader_score","finbert_score"]]

    
    twitter_weight = weighted_weekends(twitter_groupby)
    twitter_weighted = pd.DataFrame(twitter_weight, columns=["date","flair_sentiment_score","vader_score","finbert_score"])
    twitter_weighted['date'] = twitter_weighted['date'].astype(str)
    panel_data['date'] = panel_data['date'].astype(str)
    twitter_merge3 = pd.merge(twitter_weighted,panel_data, on="date")
    twitter_merge3.corr()[["flair_sentiment_score","vader_score","finbert_score"]]

    
    
    stocktwits_weight = weighted_weekends(stocktwits_groupby)
    stocktwits_weighted = pd.DataFrame(stocktwits_weight, columns=["date","flair_sentiment_score","vader_score","finbert_score"])
    stocktwits_weighted['date'] = stocktwits_weighted['date'].astype(str)
    panel_data['date'] = panel_data['date'].astype(str)
    stocktwits_merge3 = pd.merge(stocktwits_weighted,panel_data, on="date")
    stocktwits_merge3.corr()[["flair_sentiment_score","vader_score","finbert_score"]]

    news_weight = weighted_weekends(news_groupby)
    news_weighted = pd.DataFrame(news_weight, columns=["date","flair_sentiment_score","vader_score","finbert_score"])
    news_weighted['date'] = news_weighted['date'].astype(str)
    panel_data['date'] = panel_data['date'].astype(str)
    news_merge3 = pd.merge(news_weighted,panel_data, on="date")
    news_merge3.corr()[["flair_sentiment_score","vader_score","finbert_score"]]

    
    columns = ["flair_sentiment_score","vader_score","finbert_score","platform","ticket"]
    weighteddrop = []
    weighteddrop.append(reddit_merge3.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[6].tolist())
    weighteddrop.append(twitter_merge3.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[6].tolist())
    weighteddrop.append(stocktwits_merge3.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[6].tolist())
    weighteddrop.append(news_merge3.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[6].tolist())
    weighteddrop[0].append("reddit")
    weighteddrop[0].append(tickername)
    weighteddrop[1].append("twitter")
    weighteddrop[1].append(tickername)
    weighteddrop[2].append("stocktwits")
    weighteddrop[2].append(tickername)
    weighteddrop[3].append("news")
    weighteddrop[3].append(tickername)
    weighteddrop = pd.DataFrame(weighteddrop,columns=columns)

    
    return weighteddrop


 

In [130]:
i = 0
for ticket in uniqueticker:
    try:
        if i == 0:
            weighted_df = weighted_drop(ticket,reddit_data,twitter_data,stocktwits_data,news_data)
            i += 1
        else:
            weighted_df_next = weighted_drop(ticket,reddit_data,twitter_data,stocktwits_data,news_data)
            weighted_df = pd.concat([weighted_df, weighted_df_next])
    except:
        pass

    
weighted_df = weighted_df.dropna()

In [131]:
weighted_df.to_csv("correlation_weighted.csv",index=False)

In [77]:
weighted_df

,likes,flair_sentiment_score,vader_score,finbert_score,date,day_name
date,,,,,,
2020-01-05,1.000000,-0.222682,0.041850,-0.164947,2020-01-05,Sunday
2020-01-06,1.000000,-0.263133,-0.000011,-0.269296,2020-01-06,Monday
2020-01-07,1.000000,0.113954,0.060814,-0.102651,2020-01-07,Tuesday
2020-01-08,1.000000,0.078797,0.053881,-0.045471,2020-01-08,Wednesday
2020-01-09,1.290000,-0.332066,0.184554,0.004163,2020-01-09,Thursday
...,...,...,...,...,...,...
2021-12-05,3.457364,-0.560480,0.281212,0.004095,2021-12-05,Sunday
2021-12-06,2.973684,-0.515939,0.579839,0.155324,2021-12-06,Monday
2021-12-07,7.166667,-0.302233,0.415522,0.033591,2021-12-07,Tuesday


# correlation test 4 weekend equal friday price include weekends all stocks

In [160]:
# add price to weekends 


def forwardfill(tickername,reddit_data,twitter_data,stocktwits_data,news_data):

    # get panel data
    start_date = '2020-04-29'
    end_date = '2021-10-10'
    panel_data = data.DataReader(tickername,'yahoo', start_date, end_date)
    panel_data["date"] = panel_data.index
    panel_data["date"] = pd.to_datetime(panel_data['date'])

    # groupby dates then average
    reddit_groupby = reddit_data[reddit_data['ticker'] == tickername].groupby(["date"]).mean()
    twitter_groupby = twitter_data[twitter_data['ticker'] == tickername].groupby(["date"]).mean()
    stocktwits_groupby = stocktwits_data[stocktwits_data['ticker'] == tickername].groupby(["date"]).mean()
    news_groupby = news_data[news_data['ticker'] == tickername].groupby(["date"]).mean()
    

    # add price to weekends 

    stocktwits_groupby["date"] = stocktwits_groupby.index
    stocktwits_groupby["date"] = pd.to_datetime(stocktwits_groupby['date'])


    stocktwits_groupby.index.name = None
    panel_data.index.name = None
    panel_data["date"] = pd.to_datetime(panel_data['date'])

    stocktwits_merge4 = pd.merge(stocktwits_groupby,panel_data, on='date', how='left')
    stocktwits_merge4.fillna(method='ffill', inplace=True)
    stocktwits_merge4 = stocktwits_merge4.dropna()
    stocktwits_merge4.corr()[["flair_sentiment_score","vader_score","finbert_score"]]

    # add price to weekends 

    reddit_groupby["date"] = reddit_groupby.index
    reddit_groupby["date"] = pd.to_datetime(reddit_groupby['date'])

    reddit_groupby.index.name = None
    panel_data.index.name = None
    panel_data["date"] = pd.to_datetime(panel_data['date'])

    reddit_merge4 = pd.merge(reddit_groupby,panel_data, on='date', how='left')
    reddit_merge4.fillna(method='ffill', inplace=True)
    reddit_merge4 = reddit_merge4.dropna()
    reddit_merge4.corr()[["flair_sentiment_score","vader_score","finbert_score"]]

    # add price to weekends 

    twitter_groupby["date"] = twitter_groupby.index
    twitter_groupby["date"] = pd.to_datetime(twitter_groupby['date'])

    twitter_groupby.index.name = None
    panel_data.index.name = None
    panel_data["date"] = pd.to_datetime(panel_data['date'])

    twitter_merge4 = pd.merge(twitter_groupby,panel_data, on='date', how='left')
    twitter_merge4.fillna(method='ffill', inplace=True)
    twitter_merge4 = twitter_merge4.dropna()
    twitter_merge4.corr()[["flair_sentiment_score","vader_score","finbert_score"]]

    # add price to weekends 
    news_groupby["date"] = news_groupby.index
    news_groupby["date"] = pd.to_datetime(news_groupby['date'])
    news_groupby.index.name = None
    panel_data.index.name = None
    panel_data["date"] = pd.to_datetime(panel_data['date'])

    news_merge4 = pd.merge(news_groupby,panel_data, on='date', how='left')
    news_merge4.fillna(method='ffill', inplace=True)
    news_merge4 = news_merge4.dropna()

    news_merge4.corr()[["flair_sentiment_score","vader_score","finbert_score"]]
    
    columns = ["flair_sentiment_score","vader_score","finbert_score","platform","ticket"]
    forwardfill = []
    forwardfill.append(reddit_merge4.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[7].tolist())
    forwardfill.append(twitter_merge4.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[10].tolist())
    forwardfill.append(stocktwits_merge4.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[10].tolist())
    forwardfill.append(news_merge4.corr()[["flair_sentiment_score","vader_score","finbert_score"]].iloc[4].tolist())
    forwardfill[0].append("reddit")
    forwardfill[0].append(tickername)
    forwardfill[1].append("twitter")
    forwardfill[1].append(tickername)
    forwardfill[2].append("stocktwits")
    forwardfill[2].append(tickername)
    forwardfill[3].append("news")
    forwardfill[3].append(tickername)
    forwardfill = pd.DataFrame(forwardfill,columns=columns)
    
    return forwardfill

In [161]:
i = 0
for ticket in uniqueticker:
    try:
        if i == 0:
            forward_df = forwardfill(ticket,reddit_data,twitter_data,stocktwits_data,news_data)
            i += 1
        else:
            forward_df_next = forwardfill(ticket,reddit_data,twitter_data,stocktwits_data,news_data)
            forward_df = pd.concat([forward_df, forward_df_next])
    except:
        pass


In [162]:
forward_df.to_csv("correlation_forwardfill.csv",index=False)

In [164]:
forward_df

,flair_sentiment_score,vader_score,finbert_score,platform,ticket
0,-0.462680,0.490076,0.290962,reddit,AAPL
1,-0.087781,0.042162,-0.082909,twitter,AAPL
2,0.218446,0.176536,0.249079,stocktwits,AAPL
3,-0.111472,0.102864,0.004312,news,AAPL
0,-0.227035,0.245138,-0.065704,reddit,AMZN
1,-0.069158,-0.003058,-0.020888,twitter,AMZN
2,0.138336,0.107891,0.195483,stocktwits,AMZN
3,0.069131,0.003813,0.024932,news,AMZN
0,-0.026855,0.383422,0.181014,reddit,ARKK
1,0.285385,0.187050,0.152075,twitter,ARKK
